# Week 4 — ADD-only + SPC Correction Hybrid

Pipeline:
1. **PhoBERT argmax** → baseline (verify ~0.5543)
2. **ADD-only** (Component 1) → verify ~0.5568
3. **ADD-only + SPC correction** (Component 1 + 2) → target ~0.57

Chiến lược:
- Component 1 (ADD-only): LLM chỉ **thêm** aspect bị miss → bảo vệ ACD
- Component 2 (SPC correction): với aspect đã present và SPC uncertain (entropy cao) → LLM sửa sentiment
- LLM **không bao giờ** xóa aspect đã được PhoBERT detect → ACD F1 không bị giảm

In [ ]:
# Cell 1 — Kaggle GPU + dependencies
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu} | VRAM: {vram:.1f} GB')
else:
    raise RuntimeError('Không có GPU. Kaggle: Settings -> Accelerator -> GPU T4 x2')

torch.cuda.empty_cache()
print(f'PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')

!pip install -q --upgrade pip
!pip install -q transformers==4.41.2 sentence-transformers==2.7.0 tokenizers==0.19.1 \
    accelerate==0.30.1 underthesea py_vncorenlp tabulate tqdm scikit-learn sentencepiece faiss-cpu
!pip install -q openai google-generativeai
print('Dependencies installed')

In [ ]:
# Cell 2 — Clone/pull latest repo
import os, sys

REPO_URL    = 'https://github.com/vudinhminh08/NLP-project-master-study.git'
REPO_BRANCH = 'master'
PROJECT_DIR = '/kaggle/working/absa-project'

if not os.path.exists(PROJECT_DIR):
    !git clone --branch {REPO_BRANCH} --depth=1 {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull origin {REPO_BRANCH}

os.chdir(PROJECT_DIR)
print(f'Working dir: {os.getcwd()}')

for p in ['code/week1', 'code/week2', 'code/week3', 'code/week3_part2', 'code/week4']:
    full = os.path.join(PROJECT_DIR, p)
    if full not in sys.path:
        sys.path.insert(0, full)

print('sys.path updated with week4')

In [ ]:
# Cell 3 — Load API key từ Kaggle Secrets
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
OPENAI_API_KEY = secrets.get_secret('OPENAI_API_KEY')
print('OPENAI_API_KEY loaded')

# Nếu dùng Gemini thay vì OpenAI:
# GEMINI_API_KEY = secrets.get_secret('GEMINI_API_KEY')
# LLM_PROVIDER = 'gemini'
# LLM_API_KEY = GEMINI_API_KEY

LLM_PROVIDER = 'openai'
LLM_API_KEY  = OPENAI_API_KEY

In [ ]:
# Cell 4 — Verify required files
import os, shutil
from glob import glob

CHECKPOINT_PATH    = 'outputs/results/week2_results_VNcoreNLP/models_cls_only/best_model.pt'
EMBEDDINGS_CACHE   = 'outputs/results/embeddings_cache.npy'
KAGGLE_DATASET_NAME = 'best_model_pt_phobertv2'   # tên Kaggle Dataset chứa checkpoint

def copy_first_match(patterns, dst, required=False):
    if os.path.exists(dst):
        return dst
    for pat in patterns:
        matches = glob(pat, recursive=True)
        if matches:
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy(matches[0], dst)
            print(f'[COPIED] {matches[0]} -> {dst}')
            return dst
    if required:
        print(f'[MISSING] {dst}')
    return None

copy_first_match(
    [f'/kaggle/input/{KAGGLE_DATASET_NAME}/best_model.pt',
     f'/kaggle/input/{KAGGLE_DATASET_NAME}/**/best_model.pt',
     '/kaggle/input/**/models_cls_only/best_model.pt'],
    CHECKPOINT_PATH, required=True,
)
copy_first_match(
    ['/kaggle/input/**/embeddings_cache.npy'],
    EMBEDDINGS_CACHE, required=False,
)

required = [
    'data/train_preprocessed.csv',
    'data/dev_preprocessed.csv',
    'data/test_preprocessed.csv',
    'outputs/eda/class_weights.json',
    CHECKPOINT_PATH,
]
for f in required:
    size = os.path.getsize(f) if os.path.exists(f) else 0
    print(f'[{"OK" if os.path.exists(f) else "MISSING"}] {f} ({size:,} bytes)')

missing = [f for f in required if not os.path.exists(f)]
assert not missing, f'Missing: {missing}'
print('All required files verified.')

In [ ]:
# Cell 5 — Setup: load data, model, retriever, LLM
import json, torch
import pandas as pd
from transformers import AutoTokenizer

from utils.constants import PHOBERT_V2, TRAIN_CONFIG, ZERO_TRAIN_ASPECTS, ASPECT_COLUMNS
from utils.helpers import set_seed, save_json
from step2_dataloader import ABSADataset, get_dataloader
from model import ABSAPhoBERT
from predict import load_best_model
from rag_retriever import ABSARetriever
from llm_client import LLMClient
from add_spc_predictor import AddSPCPredictor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_seed(TRAIN_CONFIG['seed'])

# Load data
train_df = pd.read_csv('data/train_preprocessed.csv')
dev_df   = pd.read_csv('data/dev_preprocessed.csv')
test_df  = pd.read_csv('data/test_preprocessed.csv')

class_weights = json.load(open('outputs/eda/class_weights.json'))
tokenizer = AutoTokenizer.from_pretrained(PHOBERT_V2)

dev_loader  = get_dataloader(dev_df,  tokenizer, batch_size=32, shuffle=False)
test_loader = get_dataloader(test_df, tokenizer, batch_size=32, shuffle=False)

# Load model
model = ABSAPhoBERT(
    model_name=PHOBERT_V2,
    dropout=TRAIN_CONFIG['dropout'],
    encoder_option='cls_only',
).to(device)
model = load_best_model(CHECKPOINT_PATH, model, device)

# RAG retriever
retriever = ABSARetriever(cache_path=EMBEDDINGS_CACHE, use_faiss=True)
retriever.fit(train_df)

# LLM client
llm_client = LLMClient(provider=LLM_PROVIDER, api_key=LLM_API_KEY)

# Predictor
predictor = AddSPCPredictor(
    model=model,
    train_df=train_df,
    retriever=retriever,
    llm_client=llm_client,
    device=device,
    k_add=4,
    k_spc=4,
    max_spc_per_review=5,
    sleep_sec=0.5,
)

print(f'Device: {device}')
print(f'Train: {len(train_df)} | Dev: {len(dev_df)} | Test: {len(test_df)}')
print(f'Embeddings cache: {EMBEDDINGS_CACHE} | exists={os.path.exists(EMBEDDINGS_CACHE)}')

In [ ]:
# Cell 6 — Smoke test: 10 reviews, tất cả 3 variants
from step4_eval import evaluate_predictions

SMOKE_N = 10
EXCLUDE = ZERO_TRAIN_ASPECTS

print('=== Smoke test (10 reviews) ===')

# Variant C (ADD+SPC) trên 10 reviews
y_true_s, preds_s, stats_s = predictor.predict(
    test_df=test_df,
    test_loader=test_loader,
    add_threshold=0.08,
    spc_entropy_threshold=0.8,
    max_samples=SMOKE_N,
    run_add_only=True,
    run_spc_correction=True,
)
smoke_metrics = evaluate_predictions(
    y_true_s, preds_s,
    title='Smoke — ADD+SPC (10 reviews)',
    exclude_aspects=EXCLUDE,
)

# In SPC correction stats
spc_s = stats_s.get('spc_stats', {})
print(f"  SPC routing rate:  {spc_s.get('routing_rate', 0):.0%}")
print(f"  SPC total changed: {spc_s.get('total_changed', 0)}")
print(f"  SPC parse fail:    {spc_s.get('parse_fail_rate', 0):.0%}")
print(f"  Smoke Combined F1: {smoke_metrics['macro_combined_f1']:.4f}")
print('Smoke test OK — proceed to full run')

In [ ]:
# Cell 7 — Tune SPC entropy threshold trên dev set
# (bỏ qua nếu muốn dùng default=0.8)

TUNE_ON_DEV = True
TUNE_MAX_SAMPLES = 200   # subset cho nhanh

if TUNE_ON_DEV:
    print('Tuning SPC entropy threshold on dev set...')
    best_threshold, sweep_results = predictor.tune_spc_threshold(
        dev_df=dev_df,
        dev_loader=dev_loader,
        add_threshold=0.08,
        entropy_thresholds=[0.3, 0.5, 0.7, 0.9, 1.1, 1.3],
        exclude_aspects=ZERO_TRAIN_ASPECTS,
        max_samples=TUNE_MAX_SAMPLES,
    )
    os.makedirs('outputs/results/week4', exist_ok=True)
    save_json(
        {'best_threshold': best_threshold, 'sweep': sweep_results},
        'outputs/results/week4/threshold_sweep_results.json',
    )
    print(f'Best entropy_threshold: {best_threshold}')
else:
    best_threshold = 0.8
    print(f'Using default entropy_threshold: {best_threshold}')

In [ ]:
# Cell 8 — Full run: 3 variants trên toàn bộ test set
import numpy as np
from hybrid_predictor import collect_logits_and_preds

os.makedirs('outputs/results/week4', exist_ok=True)
EXCLUDE = ZERO_TRAIN_ASPECTS

# --- Variant A: PhoBERT argmax baseline ---
print('\n=== Variant A: PhoBERT argmax baseline ===')
all_probs, argmax_preds, y_true = collect_logits_and_preds(model, test_loader, device)
baseline_metrics = evaluate_predictions(
    y_true, argmax_preds,
    title='A — PhoBERT argmax',
    exclude_aspects=EXCLUDE,
    save_path='outputs/results/week4/baseline_metrics.json',
)
print(f"  Combined F1: {baseline_metrics['macro_combined_f1']:.4f} "
      f"| ACD: {baseline_metrics['macro_acd_f1']:.4f} "
      f"| SPC: {baseline_metrics['macro_spc_f1']:.4f}")

# --- Variant B: ADD-only ---
print('\n=== Variant B: ADD-only (Component 1) ===')
y_true_b, preds_b, stats_b = predictor.predict(
    test_df=test_df, test_loader=test_loader,
    add_threshold=0.08, spc_entropy_threshold=999.0,
    run_add_only=True, run_spc_correction=False,
)
add_only_metrics = evaluate_predictions(
    y_true_b, preds_b,
    title='B — ADD-only',
    exclude_aspects=EXCLUDE,
    save_path='outputs/results/week4/add_only_metrics.json',
)
add_stats = stats_b.get('add_stats', {})
print(f"  Combined F1: {add_only_metrics['macro_combined_f1']:.4f} "
      f"| ACD: {add_only_metrics['macro_acd_f1']:.4f} "
      f"| SPC: {add_only_metrics['macro_spc_f1']:.4f}")
print(f"  ADD trigger rate: {add_stats.get('trigger_rate', 0):.0%} "
      f"| Total added: {add_stats.get('total_added', 0)}")

# --- Variant C: ADD-only + SPC correction ---
print(f'\n=== Variant C: ADD-only + SPC correction (entropy_threshold={best_threshold}) ===')
y_true_c, preds_c, stats_c = predictor.predict(
    test_df=test_df, test_loader=test_loader,
    add_threshold=0.08,
    spc_entropy_threshold=best_threshold,
    run_add_only=True, run_spc_correction=True,
)
add_spc_metrics = evaluate_predictions(
    y_true_c, preds_c,
    title='C — ADD-only + SPC correction',
    exclude_aspects=EXCLUDE,
    save_path='outputs/results/week4/add_spc_metrics.json',
)
spc_stats = stats_c.get('spc_stats', {})
print(f"  Combined F1: {add_spc_metrics['macro_combined_f1']:.4f} "
      f"| ACD: {add_spc_metrics['macro_acd_f1']:.4f} "
      f"| SPC: {add_spc_metrics['macro_spc_f1']:.4f}")
print(f"  SPC routing rate: {spc_stats.get('routing_rate', 0):.0%} "
      f"| Changed: {spc_stats.get('total_changed', 0)} "
      f"| Parse fail: {spc_stats.get('parse_fail_rate', 0):.0%}")

# Save SPC log
if spc_stats and not spc_stats.get('skipped'):
    save_json(spc_stats, 'outputs/results/week4/spc_correction_log.json')

In [ ]:
# Cell 9 — Ablation table + so sánh với cascade v1.5
from tabulate import tabulate

# Load cascade v1.5 reference nếu có
CASCADE_REF = {
    'cascade_v1_5': {'acd_f1': 0.6260, 'spc_f1': 0.5019, 'combined_f1': 0.5639},
    'old_ensemble_threshold': {'acd_f1': None, 'spc_f1': None, 'combined_f1': 0.5644},
}

rows = [
    ['A — PhoBERT baseline',
     f"{baseline_metrics['macro_acd_f1']:.4f}",
     f"{baseline_metrics['macro_spc_f1']:.4f}",
     f"{baseline_metrics['macro_combined_f1']:.4f}"],
    ['B — ADD-only',
     f"{add_only_metrics['macro_acd_f1']:.4f}",
     f"{add_only_metrics['macro_spc_f1']:.4f}",
     f"{add_only_metrics['macro_combined_f1']:.4f}"],
    ['C — ADD + SPC correction (ours)',
     f"{add_spc_metrics['macro_acd_f1']:.4f}",
     f"{add_spc_metrics['macro_spc_f1']:.4f}",
     f"{add_spc_metrics['macro_combined_f1']:.4f}"],
    ['(ref) Cascade v1.5',
     f"{CASCADE_REF['cascade_v1_5']['acd_f1']:.4f}",
     f"{CASCADE_REF['cascade_v1_5']['spc_f1']:.4f}",
     f"{CASCADE_REF['cascade_v1_5']['combined_f1']:.4f}"],
    ['(ref) Old ensemble+threshold',
     '—', '—',
     f"{CASCADE_REF['old_ensemble_threshold']['combined_f1']:.4f}"],
]

print(tabulate(rows, headers=['Variant', 'ACD F1', 'SPC F1', 'Combined F1'], tablefmt='github'))

# Save summary
summary = {
    'A_baseline':  {'acd_f1': baseline_metrics['macro_acd_f1'],  'spc_f1': baseline_metrics['macro_spc_f1'],  'combined_f1': baseline_metrics['macro_combined_f1']},
    'B_add_only':  {'acd_f1': add_only_metrics['macro_acd_f1'],  'spc_f1': add_only_metrics['macro_spc_f1'],  'combined_f1': add_only_metrics['macro_combined_f1']},
    'C_add_spc':   {'acd_f1': add_spc_metrics['macro_acd_f1'],   'spc_f1': add_spc_metrics['macro_spc_f1'],   'combined_f1': add_spc_metrics['macro_combined_f1'],
                   'spc_entropy_threshold': best_threshold,
                   'routing_rate': spc_stats.get('routing_rate', 0),
                   'total_changed': spc_stats.get('total_changed', 0),
                   'parse_fail_rate': spc_stats.get('parse_fail_rate', 0)},
    'llm_usage':   llm_client.get_usage_stats(),
}
save_json(summary, 'outputs/results/week4/week4_summary.json')
print('\nSaved: outputs/results/week4/week4_summary.json')